In [1]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from pypdf import PdfReader

In [16]:
# Setup
API_KEY = "not-needed"
BASE_URL = "http://localhost:11434/v1"
MODEL = "gpt-oss:20b"

In [23]:
# Creating the system prompt
system_message = """Du bist ein Assistent der Textparagraphen aus einem PDF Dokument extrahiert, die relevant für 
Beantwortung von vom Benutzer eingegebenen Fragen sind. Folge dabei folgenden Regeln:
* Verwende nur Informationen aus dem hochgeladenen PDF Dokument.
* Erfinde keine Informationen, sondern gib nur relevanten Text wie er im PDF Dokument geschrieben steht zurück.
* Fasse nichts zusammen und verändere den Text nicht. 
* Wenn ein Paragraph für sich alleine nicht verständlich ist, gib als Kontext auch umliegende Paragraphen aus.
* Gib an von welcher Seite des Dokuments der Text stammt.
"""

In [18]:
# Extract text from pdf
def extract_pdf_text(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PdfReader(file)
            for page_num, page in enumerate(pdf_reader.pages, start=1):
                page_text = page.extract_text()
                text += f"--- Seite {page_num} ---\n{page_text}\n"
            return text.strip()
    except FileNotFoundError:
        return f"File '{pdf_path}' not found."
    except Exception as e:
        return f"Error reading PDF: {str(e)}"

In [20]:
# Call model function
def call_model(prompts, api_key=API_KEY, base_url=BASE_URL, model=MODEL):
    client = OpenAI(api_key=api_key, base_url=base_url)

    response = client.chat.completions.create(
        model=model,
        messages=prompts
    )

    return response.choices[0].message.content    

In [27]:
# Chat generator function for gradio
def chat(message, history, request: gr.Request, pdf_path=None, question_input=None, system_message=system_message):
    pdf_text = None
    if pdf_path:
        pdf_text = extract_pdf_text(pdf_path)
        system_message += f"\n Hier ist der Inhalt des PDF Dokuments:\n{pdf_text}"

    if question_input:
        system_message += f"\n Extrahiere alle Textparagraphen aus dem PDF die relevant sind für folgende Frage:\n{question_input}"
    
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    return call_model(prompts=messages)

In [30]:
# Gradio chat interface
# gr.ChatInterface(fn=chat, additional_inputs=[gr.File(label="Upload PDF"), gr.Textbox(label="Deine Frage")], type="messages").launch()

# with gr.Blocks() as demo:
#     gr.Markdown("## Extrahiere relevante Textparagraphen aus PDF Dokument")

#     with gr.Row():
#         file_uploader = gr.File(label="Upload PDF", file_types=[".pdf"])
#         upload_output = gr.Textbox(label="Upload Status")

#     file_uploader.upload(extract_pdf_text, inputs=file_uploader, outputs=upload_output)

#     gr.ChatInterface(
#         fn=chat,
#         title="Ask your PDF",
#         type="messages",
#         textbox=gr.Textbox(placeholder="Welche Frage soll beantwortet werden?")
#     )

# demo.launch()

# with gr.Blocks() as demo:
#     gr.Markdown("## 📑 PDF Q&A mit GPT-OSS:20B (lokal über Ollama)")

#     with gr.Row():
#         pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
#         question_input = gr.Textbox(label="Deine Frage")

#     output = gr.Textbox(label="Antwort")

#     submit_btn = gr.Button("Frage stellen")
#     submit_btn.click(fn=chat, inputs=[pdf_input, question_input], outputs=output)

# demo.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/opt/anaconda3/envs/llms/lib/python3.11/site-packages/gradio/queueing.py", line 667, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/llms/lib/python3.11/site-packages/gradio/route_utils.py", line 349, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/llms/lib/python3.11/site-packages/gradio/blocks.py", line 2274, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/llms/lib/python3.11/site-packages/gradio/blocks.py", line 1781, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/llms/lib/python3.11/site-packages/anyio/to_thread.py", line 56, in run_sync
    return